# scarlatti-doodle project!

## Zip 파일 메모리에 로딩
#### this code block was written using AI

In [1]:
import io
import os
import zipfile
import partitura as pt  # 🔥 music21 대신 partitura 도입

def mount_scarlatti():
    """scarlatti.zip을 가상 메모리에 마운트하고, 파일 목록을 반환합니다."""
    ZIP_FILE_PATH = "original_midi.zip"
    
    if not os.path.exists(ZIP_FILE_PATH):
        raise FileNotFoundError(f"⚠️ '{ZIP_FILE_PATH}' 파일이 없습니다. 왼쪽 탐색기에 업로드해 주세요!")
        
    print("🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...")
    
    # 파일을 RAM으로 읽어오기
    with open(ZIP_FILE_PATH, "rb") as f:
        zip_buffer = io.BytesIO(f.read())
        
    # Zip 아카이브 객체 생성
    archive = zipfile.ZipFile(zip_buffer)
    
    # 미디 파일 목록만 싹 긁어오기
    midi_files = [f for f in archive.namelist() if f.lower().endswith(('.mid', '.midi'))]
    print(f"📦 마운트 완료! 총 {len(midi_files)}개의 가상 미디 파일 준비 완료.")
    
    return archive, midi_files

def load_midi_from_virtual_folder(archive, file_path):
    """
    [수정 완료] 가상 폴더(archive)와 파일 경로를 주면, 
    디스크 IO 없이 partitura의 PerformedPart 객체로 즉시 변환합니다.
    """
    # 1. 압축 파일 내부에서 순수 바이트(Bytes) 데이터 추출
    midi_raw_bytes = archive.read(file_path)
    
    # 2. BytesIO로 스트림을 감싸서 partitura가 날것 그대로 읽게 만듦
    midi_file_stream = io.BytesIO(midi_raw_bytes)
    
    # 3. music21 대신 partitura로 파싱 (첫 번째 연주 트랙 [0] 반환)
    performance = pt.load_performance_midi(midi_file_stream)[0]
    
    return performance

# 🔥 [실행] 가상 폴더 연결하기
scarlatti_folder, file_list = mount_scarlatti()
print("앞으로 scarlatti_folder, file_list에 접근해서 사용! load_midi_from_virtual_folder 함수 사용")



/home/codespace/.local/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...
📦 마운트 완료! 총 555개의 가상 미디 파일 준비 완료.
앞으로 scarlatti_folder, file_list에 접근해서 사용! load_midi_from_virtual_folder 함수 사용


#### 전처리 함수들 준비

In [7]:
from music21 import *
import partitura as pt
import numpy as np

def Score_TrebleBassSeparation_Partitura(inputScore: pt.performance.PerformedPart, splitPoint: int = 60):
    '''NoteArray 분리 후, 저장 가능한 PerformedPart 객체로 복원하여 리턴하는 함수'''
    performance = inputScore[0]
    note_array = performance.note_array()
    
    split_pitch = splitPoint
    
    treble_mask = note_array['pitch'] >= split_pitch
    bass_mask = note_array['pitch'] < split_pitch
    
    treble_note_array = note_array[treble_mask] # 이거는 단순 마스크! 프린트하면 true와 false 로 이루어진 어레이가 나옴!
    bass_note_array = note_array[bass_mask]
    
    # NumPy 배열을 partitura가 저장할 수 있는 PerformedPart 객체 상자에 다시 담아줍니다.
    treble_part = pt.performance.PerformedPart.from_note_array(treble_note_array)
    bass_part = pt.performance.PerformedPart.from_note_array(bass_note_array)
    
    return treble_part, bass_part


def Score_TransposeToAllKeys(inputScore : pt.performance.PerformedPart):
    '''스코어 파일을 모든 조로 전조해서 리턴'''
    key = pt.musicanalysis.estimate_key(inputScore)
    print(key)

    pass

def Score_SliceByMeasures(inputScore):
    '''스코어 파일을 특정 마디 길이만큼 나눠서 리턴'''
    # 미디를 16마디(32마디로 할까?)로 나눔, 0에서 2 사이의 마디만큼 겹침, 남은 마디가 부족하면 겹쳐서라도 16마디 맟춤

    pass


def Score_MaskNotes(inputScore):
    '''멜로디 데이터에서 일부 데이터들 마스킹해서 리턴'''
    pass

def ScoreToDataset(inputScore):
    '''스코어 파일을 ai 학습용 데이터셋으로 변환해서 리턴, 코드 정보와 박자정보 추가'''
    # 코드랑 박자정보 추가는 할지 말지 모르겠다...
    pass

    


In [14]:
from music21 import *

# 1. 함수 실행 및 완벽히 정제된 NumPy 배열 2개 받아오기
testFile = pt.load_performance_midi("sonatas_k-531_(c)sankey.mid")

treble_data, bass_data = Score_TrebleBassSeparation_Partitura(
    inputScore= testFile
)

# 2. 결과물 확인을 위해 각각 개별 미디 파일로 안전하게 저장하기
pt.save_performance_midi(treble_data, "partitura_treble_output.mid")
pt.save_performance_midi(bass_data, "partitura_bass_output.mid")

print("✨ 파르티투라 분리 완료! 박자가 완벽하게 고정된 미디 파일이 생성되었습니다.")

Score_TransposeToAllKeys(testFile)
s = converter.parse("sonatas_k-531_(c)sankey.mid")
print(s.analyze('key'))



✨ 파르티투라 분리 완료! 박자가 완벽하게 고정된 미디 파일이 생성되었습니다.
Bm
e minor
